## Load data

In [1]:
import pandas as pd
import re
from nltk.corpus import stopwords

df = pd.read_csv('data/customer_support_tickets.csv')
df.shape

(8469, 17)

In [2]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


## Check missing values and target columns

In [3]:
df.isnull().sum()

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

In [4]:
df['Ticket Type'].value_counts()

Ticket Type
Refund request          1752
Technical issue         1747
Cancellation request    1695
Product inquiry         1641
Billing inquiry         1634
Name: count, dtype: int64

In [5]:
df['Ticket Priority'].value_counts()

Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

## Combine subject and description into one text field

In [6]:
df['text'] = df['Ticket Subject'] + ' ' + df['Ticket Description']
df[['Ticket Subject', 'Ticket Description', 'text']].head()

,Ticket Subject,Ticket Description,text
0,Product setup,I'm having an issue with the {product_purchase...,Product setup I'm having an issue with the {pr...
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Peripheral compatibility I'm having an issue w...
2,Network problem,I'm facing a problem with my {product_purchase...,Network problem I'm facing a problem with my {...
3,Account access,I'm having an issue with the {product_purchase...,Account access I'm having an issue with the {p...
4,Data loss,I'm having an issue with the {product_purchase...,Data loss I'm having an issue with the {produc...


## Clean the text

In [7]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'{.*?}', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return ' '.join(words)

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()

,text,clean_text
0,Product setup I'm having an issue with the {pr...,product setup im issue please assist billing z...
1,Peripheral compatibility I'm having an issue w...,peripheral compatibility im issue please assis...
2,Network problem I'm facing a problem with my {...,network problem im facing problem turning work...
3,Account access I'm having an issue with the {p...,account access im issue please assist problem ...
4,Data loss I'm having an issue with the {produc...,data loss im issue please assist note seller r...


## Keep only the columns we need going forward

In [8]:
final_df = df[['Ticket ID', 'clean_text', 'Ticket Type', 'Ticket Priority']].copy()
final_df.columns = ['ticket_id', 'clean_text', 'category', 'priority']
final_df.head()

,ticket_id,clean_text,category,priority
0,1,product setup im issue please assist billing z...,Technical issue,Critical
1,2,peripheral compatibility im issue please assis...,Technical issue,Critical
2,3,network problem im facing problem turning work...,Technical issue,Low
3,4,account access im issue please assist problem ...,Billing inquiry,Low
4,5,data loss im issue please assist note seller r...,Billing inquiry,Low


In [9]:
final_df.isnull().sum()

ticket_id     0
clean_text    0
category      0
priority      0
dtype: int64

## Save cleaned data

In [10]:
final_df.to_csv('outputs/cleaned_tickets.csv', index=False)
print('saved', final_df.shape)

saved (8469, 4)
